In [ ]:
# ==============================================================
# DAILY BIAS CORRECTION 
# Final = (CHIRPS × Grid) → Scaled to monthly corrected rasters
# ==============================================================

import os, warnings, logging
from glob import glob
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import rowcol
from scipy.ndimage import gaussian_filter
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

# -------------------------- CONFIG (YOUR PATHS) --------------------------
BASE = r"D:\Working_Directory"

csv_dir       = os.path.join(BASE, "Timeseries")
tif_dir       = os.path.join(BASE, "Daily_CHIRPS")
monthly_dir   = os.path.join(BASE, "monthly_Corrected")  # <-- Your monthly corrected rasters
station_coords = os.path.join(BASE, "Coordinates.csv")
ref_raster     = glob(os.path.join(tif_dir, "P_CHIRPS.v2.0_mm-day-1_daily_2012.01.01.tif"))[0]

output_root = os.path.join(BASE, "OUTPUT")
inter_dir   = os.path.join(output_root, "intermediate")
final_dir   = os.path.join(output_root, "final_corrected")

os.makedirs(output_root, exist_ok=True)
os.makedirs(inter_dir,    exist_ok=True)
os.makedirs(final_dir,    exist_ok=True)

log_file = os.path.join(output_root, "daily_bias_correction.log")
logging.basicConfig(filename=log_file, level=logging.INFO,
                    format="%(asctime)s - %(levelname)s - %(message)s")
print(f"Log: {log_file}")
logging.info("=== DAILY BIAS CORRECTION + MONTHLY ANCHOR STARTED ===")

# -------------------------- 1. REFERENCE RASTER --------------------------
with rasterio.open(ref_raster) as src:
    meta = src.meta.copy()
    H, W = src.height, src.width
    tr = src.transform
    nodata = src.nodata
meta.update(dtype="float32", nodata=nodata)

cols, rows = np.meshgrid(np.arange(W), np.arange(H))
xs, ys = rasterio.transform.xy(tr, rows, cols)
grid_pts = np.column_stack([np.ravel(xs), np.ravel(ys)])

# -------------------------- 2. IDW FUNCTION --------------------------
def idw(coords, values, targets, power=2):
    coords = np.asarray(coords)
    values = np.asarray(values)
    targets = np.asarray(targets)
    if coords.shape[0] == 0: return np.ones(targets.shape[0], dtype=np.float32)
    dist = np.sqrt(((targets[:, None, :] - coords[None, :, :]) ** 2).sum(axis=2))
    dist[dist == 0] = 1e-12
    w = 1.0 / dist ** power
    interpolated = (w * values[None, :]).sum(axis=1) / w.sum(axis=1)
    return interpolated.astype(np.float32)

# -------------------------- 3. TRAIN RF PER STATION --------------------------
stations_df = pd.read_csv(station_coords)
station_names = stations_df["Station"].tolist()

csv_files = glob(os.path.join(csv_dir, "*.csv"))
models = {}
station_dfs = {}

for fp in csv_files:
    st = os.path.splitext(os.path.basename(fp))[0]
    try:
        df = pd.read_csv(fp, parse_dates=["Date"])
        df = df.dropna(subset=["InSitu", "WaPOR"])
        if df.empty: continue
        df = df.set_index("Date")
        df["DOY"] = df.index.dayofyear

        X = df[["WaPOR", "DOY"]].values
        y = df["InSitu"].values
        if len(X) < 5: continue

        rf = RandomForestRegressor(n_estimators=300, random_state=42)
        rf.fit(X, y)
        models[st] = rf
        df["Corrected"] = rf.predict(X)
        station_dfs[st] = df

        out_csv = os.path.join(output_root, f"{st}_corrected.csv")
        df[["InSitu", "WaPOR", "Corrected"]].reset_index().to_csv(out_csv, index=False)
        logging.info(f"Saved: {out_csv}")
    except Exception as e:
        logging.error(f"Failed {st}: {e}")

if not models:
    raise ValueError("No valid station data!")

logging.info(f"Trained {len(models)} RF models")

# -------------------------- 4. DAILY CORRECTION FACTORS --------------------------
daily_factors = defaultdict(dict)

for st, df in station_dfs.items():
    for date_str in df.index.strftime("%Y-%m-%d"):
        try:
            ts = pd.Timestamp(date_str)
            row = df.loc[ts]
            wapor = row["WaPOR"]
            corrected = row["Corrected"]
            factor = corrected / wapor if wapor > 0 else 1.0
            daily_factors[date_str][st] = float(factor)
        except:
            continue

factor_df = pd.DataFrame([
    {"Date": d, "Station": s, "Correction_Factor": f}
    for d, sdict in daily_factors.items() for s, f in sdict.items()
])
factor_df.to_csv(os.path.join(output_root, "daily_correction_factors.csv"), index=False)

# -------------------------- 5. FIRST PASS: CHIRPS × GRID → INTERMEDIATE --------------------------
tif_files = sorted(glob(os.path.join(tif_dir, "*.tif")))
logging.info(f"Processing {len(tif_files)} CHIRPS rasters")

for tif_path in tif_files:
    fname = os.path.basename(tif_path)
    try:
        date_str = fname.split("_")[-1].replace(".tif", "")
        dt = datetime.strptime(date_str, "%Y.%m.%d")
    except:
        logging.warning(f"Skip: {fname}")
        continue
    ymd = dt.strftime("%Y-%m-%d")

    # Build factor grid
    day_dict = daily_factors.get(ymd, {})
    active = [s for s in station_names if s in day_dict]
    if len(active) == 0:
        factor_grid = np.ones((H, W), dtype=np.float32)
    else:
        lon_sub = stations_df[stations_df["Station"].isin(active)]["Longitude"].values
        lat_sub = stations_df[stations_df["Station"].isin(active)]["Latitude"].values
        fac_sub = np.array([day_dict[s] for s in active])
        interpolated = idw(np.column_stack([lon_sub, lat_sub]), fac_sub, grid_pts)
        factor_grid = interpolated.reshape((H, W)).astype(np.float32)
        factor_grid = gaussian_filter(factor_grid, sigma=1.5)

    # Save correction grid
    grid_path = os.path.join(output_root, f"CorrectionGrid_{date_str}_rf_idw_smoothed.tif")
    with rasterio.open(grid_path, "w", **meta) as dst:
        dst.write(factor_grid, 1)

    # Apply to CHIRPS
    with rasterio.open(tif_path) as src:
        chirps = src.read(1).astype(np.float32)
        msk = chirps == nodata
    corrected = chirps * factor_grid
    corrected[msk] = nodata

    # Save intermediate
    inter_path = os.path.join(inter_dir, fname.replace(".tif", "_inter.tif"))
    with rasterio.open(inter_path, "w", **meta) as dst:
        dst.write(corrected, 1)

# -------------------------- 6. MONTHLY ANCHORING → FINAL --------------------------
# Load monthly rasters

# -------------------------- PATHS --------------------------
BASE = r"D:\Working_Directory"

monthly_dir = os.path.join(BASE, "monthly_Corrected")
inter_dir   = os.path.join(BASE, "OUTPUT", "intermediate")
final_dir   = os.path.join(BASE, "OUTPUT", "final_corrected")

os.makedirs(final_dir, exist_ok=True)

log_file = os.path.join(BASE, "OUTPUT", "anchoring_final.log")
logging.basicConfig(filename=log_file, level=logging.INFO,
                    format="%(asctime)s - %(levelname)s - %(message)s")
print(f"Log: {log_file}")
logging.info("=== FINAL ANCHORING STARTED ===")

# -------------------------- MONTHLY RASTERS (YYYY.MM) --------------------------
monthly_files = sorted(glob.glob(os.path.join(monthly_dir, "*_corrected_*.tif")))
print(f"\nFound {len(monthly_files)} monthly files")
monthly_by_ym = {}
pattern = re.compile(r"_(\d{4})\.(\d{2})\.(\d{2})\.tif$")   # _2012.01.01.tif

for p in monthly_files:
    bn = os.path.basename(p)
    m = pattern.search(bn)
    if m:
        y, m, d = map(int, m.groups())
        monthly_by_ym[(y, m)] = p
        print(f"  Monthly {y}-{m:02d} → {bn}")
    else:
        print(f"  SKIP malformed monthly: {bn}")

if not monthly_by_ym:
    raise ValueError("No monthly rasters parsed!")

# -------------------------- INTERMEDIATE DAILY (YYYY.MM.DD) --------------------------
inter_files = sorted(glob.glob(os.path.join(inter_dir, "*_inter.tif")))
print(f"\nFound {len(inter_files)} intermediate files")
daily_by_ym = defaultdict(list)

for fp in inter_files:
    bn = os.path.basename(fp).replace("_inter.tif", "")
    m = pattern.search(bn + ".tif")   # fake .tif to match pattern
    if m:
        y, m, d = map(int, m.groups())
        daily_by_ym[(y, m)].append(fp)
        print(f"  Grouped {y}-{m:02d} ← {bn}")
    else:
        print(f"  SKIP malformed daily: {bn}")

if not daily_by_ym:
    raise ValueError("No intermediate files grouped!")

# -------------------------- ANCHORING LOOP --------------------------
final_count = 0
for (y, m), dpaths in daily_by_ym.items():
    mon_path = monthly_by_ym.get((y, m))
    if not mon_path:
        print(f"  No monthly for {y}-{m:02d}")
        continue

    # Load monthly total
    with rasterio.open(mon_path) as src:
        mon_total = src.read(1).astype(np.float32)
        nodata = src.nodata
        mon_total[mon_total == nodata] = np.nan

    # Sum daily intermediates
    daily_sum = np.zeros_like(mon_total)
    for dp in dpaths:
        with rasterio.open(dp) as src:
            arr = src.read(1).astype(np.float32)
            arr[arr == nodata] = 0.0
            daily_sum += arr

    # Scale factor
    scale = np.ones_like(daily_sum)
    valid = (daily_sum > 0) & (~np.isnan(mon_total))
    if valid.any():
        scale[valid] = mon_total[valid] / daily_sum[valid]

    # Write each final raster
    for dp in dpaths:
        bn = os.path.basename(dp).replace("_inter.tif", "")
        # Reconstruct date from filename
        m2 = pattern.search(bn + ".tif")
        if not m2:
            continue
        y2, m2, d2 = map(int, m2.groups())
        date_part = f"{y2}.{m2:02d}.{d2:02d}"
        prefix = bn.split("_" + date_part)[0]
        final_name = f"{prefix}_corrected_{date_part}.tif"
        out_path = os.path.join(final_dir, final_name)

        with rasterio.open(dp) as src:
            inter = src.read(1).astype(np.float32)
            meta = src.meta.copy()
        final = inter * scale
        final[inter == nodata] = nodata
        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(final, 1)
        final_count += 1
        print(f"  FINAL → {final_name}")

# -------------------------- FINAL CHECK --------------------------
final_files = glob.glob(os.path.join(final_dir, "*_corrected_*.tif"))
print(f"\nSUCCESS! {final_count} final rasters created")
print(f"Folder: {final_dir}")
print(f"Sample: {os.path.basename(final_files[0]) if final_files else 'None'}")

# -------------------------- 7. VALIDATION --------------------------
# ----------------------------------------------------
BASE = r"D:\Working_Directory"
csv_dir = os.path.join(BASE, "OUTPUT")  # where *_corrected.csv are saved
output_dir = csv_dir

os.makedirs(output_dir, exist_ok=True)

log_file = os.path.join(output_dir, "validation_csv.log")
logging.basicConfig(filename=log_file, level=logging.INFO,
                    format="%(asctime)s - %(levelname)s - %(message)s")
print(f"Validation log: {log_file}")
logging.info("=== CSV-ONLY VALIDATION STARTED ===")

# -------------------------- FIND ALL CORRECTED CSVs --------------------------
csv_files = sorted(glob.glob(os.path.join(csv_dir, "*_corrected.csv")))
print(f"\nFound {len(csv_files)} station CSVs")

if not csv_files:
    raise FileNotFoundError(f"No *_corrected.csv files in {csv_dir}")

# -------------------------- VALIDATION LOOP --------------------------
validation = []

for fp in csv_files:
    station = os.path.basename(fp).replace("_corrected.csv", "")
    print(f"\nProcessing: {station}")

    try:
        df = pd.read_csv(fp, parse_dates=["Date"])
        df = df.dropna(subset=["InSitu", "Corrected"])
        if len(df) < 10:
            print(f"  Skip: <10 valid days")
            continue

        obs = df["InSitu"].values
        pred = df["Corrected"].values

        # Metrics
        rmse = np.sqrt(mean_squared_error(obs, pred))
        corr, _ = pearsonr(obs, pred)
        nse = 1 - np.sum((obs - pred)**2) / np.sum((obs - np.mean(obs))**2)
        rel_bias = (np.mean(pred) - np.mean(obs)) / np.mean(obs) if np.mean(obs) != 0 else np.nan

        validation.append({
            "Station": station,
            "RMSE": round(rmse, 4),
            "Relative_Bias": round(rel_bias, 4),
            "Pearson_R": round(corr, 4),
            "NSE": round(nse, 4),
            "N_Days": len(obs)
        })

        # Scatter Plot
        plt.figure(figsize=(7, 6))
        plt.scatter(obs, pred, alpha=0.6, s=30)
        max_val = max(obs.max(), pred.max())
        plt.plot([0, max_val], [0, max_val], 'r--', lw=1.5)
        plt.xlabel("InSitu (mm/day)")
        plt.ylabel("Corrected (mm/day)")
        plt.title(f"{station}\nR={corr:.3f}, NSE={nse:.3f}, N={len(obs)}")
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"{station}_scatter.png"), dpi=150)
        plt.close()

        # Time Series Plot
        plt.figure(figsize=(12, 4))
        plt.plot(df["Date"], df["InSitu"], 'o-', label="InSitu", markersize=3)
        plt.plot(df["Date"], df["Corrected"], 's-', label="Corrected", markersize=3)
        plt.ylabel("Rainfall (mm/day)")
        plt.title(f"{station} – Daily Time Series")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"{station}_timeseries.png"), dpi=150)
        plt.close()

        logging.info(f"{station}: R={corr:.3f}, NSE={nse:.3f}, N={len(obs)}")

    except Exception as e:
        logging.error(f"Failed {station}: {e}")
        print(f"  ERROR: {e}")

# -------------------------- SAVE METRICS --------------------------
val_df = pd.DataFrame(validation)
metrics_csv = os.path.join(output_dir, "daily_validation_metrics.csv")
val_df.to_csv(metrics_csv, index=False)

print(f"\nSUCCESS! Validation complete.")
print(f"  {len(val_df)} stations validated")
print(f"  Metrics: {metrics_csv}")
print(f"  Plots saved in: {output_dir}")
print(val_df[["Station", "RMSE", "Pearson_R", "NSE", "N_Days"]])

